In [ ]:
# Phase 5: SHAP Explainability Analysis
# =======================================
# Extended SHAP analysis for CatBoost model

# %% [markdown]
# # Phase 5: SHAP Explainability Analysis
# 
# This notebook provides extended SHAP analysis building on Phase 4's foundation:
# - Global feature importance (summary plot)
# - Feature dependence plots (interactions)
# - Waterfall charts (individual predictions)
# - Feature interaction heatmap

# %% Import libraries
import sys
sys.path.append('src/evaluation')

from shap_explainer import SHAPExplainer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# %% [markdown]
# ## 1. Initialize SHAP Explainer

# %% Load model and data
explainer = SHAPExplainer(model_path="models/catboost_best.pkl")
X, y, feature_names = explainer.prepare_data()

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {len(feature_names)}")

# %% [markdown]
# ## 2. Compute SHAP Values

# %% Compute SHAP
explainer.compute_shap_values(X, feature_names)

# %% [markdown]
# ## 3. Global Feature Importance

# %% Summary plot
explainer.plot_summary()

# %% Feature importance table
importance_df = explainer.save_feature_importance()
print("\n📊 Top 20 Features:")
print(importance_df.head(20).to_string(index=False))

# %% [markdown]
# ### Interpretation
# 
# The SHAP summary plot shows:
# - **Y-axis**: Features ranked by importance (mean absolute SHAP value)
# - **X-axis**: SHAP value (impact on prediction)
# - **Color**: Feature value (red=high, blue=low)
# 
# Key insights:
# - Sentiment features (ensemble, FinBERT, variance) are highly important
# - Technical indicators (CMF, MACD, volatility) complement sentiment
# - Entity-specific sentiment (CEO, products) adds predictive power

# %% [markdown]
# ## 4. Feature Dependence Analysis

# %% Get top 5 features
top_5_features = importance_df.head(5)['feature'].tolist()
print("Top 5 features for dependence plots:")
for i, feat in enumerate(top_5_features, 1):
    print(f"{i}. {feat}")

# %% Dependence plot - Feature 1
explainer.plot_dependence(top_5_features[0], X=X)

# %% Dependence plot - Feature 2
explainer.plot_dependence(top_5_features[1], X=X)

# %% Dependence plot - Feature 3
explainer.plot_dependence(top_5_features[2], X=X)

# %% Dependence plot - Feature 4
explainer.plot_dependence(top_5_features[3], X=X)

# %% Dependence plot - Feature 5
explainer.plot_dependence(top_5_features[4], X=X)

# %% [markdown]
# ### Dependence Plot Interpretation
# 
# Dependence plots show:
# - **X-axis**: Feature value
# - **Y-axis**: SHAP value (impact on prediction)
# - **Color**: Interaction feature (automatically selected by SHAP)
# 
# Look for:
# - Linear vs non-linear relationships
# - Threshold effects
# - Feature interactions (shown by color variation)

# %% [markdown]
# ## 5. Individual Prediction Explanations (Waterfall)

# %% Waterfall - First sample
print("Explaining prediction for first sample (index 0):")
explainer.plot_waterfall(0, X)

# %% Waterfall - Middle sample
mid_idx = len(X) // 2
print(f"Explaining prediction for middle sample (index {mid_idx}):")
explainer.plot_waterfall(mid_idx, X)

# %% Waterfall - Last sample
print(f"Explaining prediction for last sample (index {len(X)-1}):")
explainer.plot_waterfall(len(X) - 1, X)

# %% [markdown]
# ### Waterfall Interpretation
# 
# Waterfall charts show:
# - **Base value**: Expected model output (average prediction)
# - **Red arrows**: Features pushing prediction higher (UP)
# - **Blue arrows**: Features pushing prediction lower (DOWN)
# - **Final prediction**: Sum of base value + all feature contributions
# 
# This explains why the model made a specific prediction for individual samples.

# %% [markdown]
# ## 6. Feature Interaction Analysis

# %% Interaction heatmap
explainer.analyze_feature_interactions(top_n=5, X=X)

# %% [markdown]
# ### Interaction Interpretation
# 
# The interaction heatmap shows correlations between SHAP values:
# - **Positive correlation**: Features work together in same direction
# - **Negative correlation**: Features counteract each other
# - **Near-zero**: Features operate independently
# 
# This helps identify feature redundancy and complementary relationships.

# %% [markdown]
# ## 7. Summary Statistics

# %% SHAP value statistics
shap_values = explainer.shap_values

print("\n📊 SHAP Value Statistics:")
print(f"Shape: {shap_values.shape}")
print(f"Mean absolute SHAP: {np.abs(shap_values).mean():.4f}")
print(f"Max positive SHAP: {shap_values.max():.4f}")
print(f"Max negative SHAP: {shap_values.min():.4f}")
print(f"Standard deviation: {shap_values.std():.4f}")

# %% Feature categories
sentiment_features = [f for f in feature_names if 'sentiment' in f.lower()]
technical_features = [f for f in feature_names if any(x in f for x in ['RSI', 'MACD', 'BB', 'ATR', 'OBV', 'CMF', 'VWAP'])]
entity_features = [f for f in feature_names if any(x in f for x in ['ceo', 'entity', 'product', 'competitor'])]

print(f"\n📊 Feature Categories:")
print(f"Sentiment features: {len(sentiment_features)}")
print(f"Technical indicators: {len(technical_features)}")
print(f"Entity features: {len(entity_features)}")

# Calculate mean importance by category
sentiment_importance = importance_df[importance_df['feature'].isin(sentiment_features)]['mean_abs_shap'].mean()
technical_importance = importance_df[importance_df['feature'].isin(technical_features)]['mean_abs_shap'].mean()
entity_importance = importance_df[importance_df['feature'].isin(entity_features)]['mean_abs_shap'].mean()

print(f"\nAverage Importance:")
print(f"Sentiment: {sentiment_importance:.4f}")
print(f"Technical: {technical_importance:.4f}")
print(f"Entity: {entity_importance:.4f}")

# %% [markdown]
# ## 8. Key Findings
# 
# **Research Insights:**
# 
# 1. **Sentiment Dominance**: Sentiment features (especially ensemble and FinBERT) are the strongest predictors
# 2. **Technical Complementarity**: Technical indicators (CMF, MACD) complement sentiment signals
# 3. **Entity Context**: CEO mentions and entity-specific sentiment add meaningful information
# 4. **Model Uncertainty**: sentiment_variance captures model disagreement and market uncertainty
# 5. **Temporal Patterns**: Lagged features (volatility_lag1, return_lag1) capture momentum
# 
# **Model Behavior:**
# - The model uses a hybrid approach: sentiment + technicals + entity context
# - Feature interactions are present (shown in dependence plots)
# - Individual predictions are explainable and interpretable
# 
# These findings validate the multi-model sentiment fusion and entity extraction approaches.

# %% [markdown]
# ## ✅ Analysis Complete
# 
# All SHAP visualizations and metrics have been saved to:
# - `results/figures/shap_plots/`
# - `results/metrics/shap_feature_importance.csv`

print("\n" + "="*60)
print("✅ SHAP EXPLAINABILITY ANALYSIS COMPLETE")
print("="*60)